# Anreichern der Buspassagierdaten

## 1. Bibliotheken importieren

In [17]:
# Standardbibliotheken
from datetime import datetime

# Drittanbieter-Bibliotheken
import pandas as pd

# PySpark-Bibliotheken
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType
from pyspark.sql.functions import col, hour, minute, lag, lead, lit, month 
from pyspark.sql.window import Window

# PySpark ML-Bibliotheken
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

## 2. Spark Session erstellen

In [18]:
# Falls noch keine Session existiert, wird sie erstellt.
spark = SparkSession.builder.appName("Bus Data Enrichment").getOrCreate()

## 3. Einlesen der Parquet-Daten

In [19]:
# Pfad zur Parquet-Datei mit den bereinigten Daten
parquet_file_path = "data/clean_data"

# Lade die Parquet-Datei in einen Spark DataFrame
data_cleaned = spark.read.parquet(parquet_file_path)

# Zeige die ersten Zeilen des geladenen DataFrames an
data_cleaned.show()

## 4. Anreicherung von zeitlichen Aggreagtion und Frequenz ankommender Busse im Intervall

In [20]:
def get_interval_index(column, interval_minutes):
    return ((hour(column) * 60 + minute(column)) / interval_minutes).cast("int")

data_with_time_intervals = data_cleaned \
    .withColumn("interval_5min", get_interval_index(col("actual_arrival"), 5)) \
    .withColumn("interval_15min", get_interval_index(col("actual_arrival"), 15)) \
    .withColumn("interval_30min", get_interval_index(col("actual_arrival"), 30)) \
    .withColumn("interval_60min", get_interval_index(col("actual_arrival"), 60))

data_with_time_intervals.show()


In [21]:
intervals = ["interval_5min", "interval_15min", "interval_30min", "interval_60min"]

for interval in intervals:
    count_col = f"bus_count_{interval}"
    
    counts = (
        data_with_time_intervals
        .select("date", "stop_name", interval, "device")  # nur nötige Spalten
        .distinct()                                      # eindeutige Devices
        .groupBy("date", "stop_name", interval)
        .agg(F.count("*").alias(count_col))
    )
    
    data_with_time_intervals = (
        data_with_time_intervals
        .join(counts, on=["date", "stop_name", interval], how="left")
    )


## 5. Anreicherung mit außerordentlichen Ereignissen und Wetterdaten

In [22]:
# Lese die außerordentlichen Ereignisdaten aus einer Parquet-Datei ein
file_path = path = "data/event_data"

event_data = spark.read.parquet(file_path)

event_data.orderBy("date", "hour").show(50)

In [23]:
# 1) Join auf date, um covid_period und holiday hinzuzufügen
covid_holiday_df = (
    event_data
    .select("date", "covid_period", "is_public_holiday", "is_school_holiday")
    .distinct()  # wenn es Mehrfach-Einträge pro date gibt
)

df_step1 = (
    data_with_time_intervals
    .join(covid_holiday_df, on="date", how="left")
)

# 2) Join auf stop_name, date und interval_60min (alias für hour), um event_type hinzuzufügen
event_type_df = (
    event_data
    .select(
        col("stop_name"),
        col("date"),
        col("hour").alias("interval_60min"),
        col("event_type")
    )
    .distinct()  # optional, um Duplikate zu vermeiden
)

df_final = (
    df_step1
    .join(event_type_df,
          on=["stop_name", "date", "interval_60min"],
          how="left")
)

# füllt für die angegebenen Spalten alle nulls mit 0
data_with_events = df_final.na.fill({'event_type': 0})

data_with_events.show(20, truncate=False)

In [24]:
# Lese die Wetterdaten aus einer Parquet-Datei ein
parquet_file_path = "data/weather_data"
weather_data = spark.read.parquet(parquet_file_path)

# Benenne die Spalte "date" im DataFrame mit außerordentlichen Ereignissen in "date2" um,
# um den späteren Join mit den Wetterdaten vorzubereiten
data_with_events = data_with_events.withColumnRenamed("date", "date2")

# Führe einen Left-Join zwischen dem DataFrame mit außerordentlichen Ereignissen und den Wetterdaten durch.
# Der Join erfolgt anhand von Datum und Stunde:
# - "date2" aus dem Ereignis-DataFrame entspricht "date" in den Wetterdaten.
# - "arrival_hour" aus dem Ereignis-DataFrame entspricht "hour" in den Wetterdaten.
# Nach dem Join werden die Hilfsspalten "date2" und "hour" entfernt.
data_with_weather = data_with_events.join(
    weather_data,
    (data_with_events["date2"] == weather_data["date"]) &
    (data_with_events["interval_60min"] == weather_data["hour"]),
    how="left"
).drop("date2", "hour")

# Zeige den finalen DataFrame mit den Wetterdaten an
data_with_weather.show()

## 7. Ermittlung und Integration der Besetzungsdaten an der vorherigen Haltestelle

In [25]:
# Definiere das Window für jede Fahrt
journey_window = Window.partitionBy("journey_id").orderBy("actual_arrival")

# Berechne Lag 1 für occupancy (bereits vorhanden)
data_with_lags = data_with_weather.withColumn(
    "occupancy_s-1", F.lag("occupancy", 1).over(journey_window)
)

# Berechne Lag 2 für occupancy und delay
data_with_lags = data_with_lags \
    .withColumn("occupancy_s-2", F.lag("occupancy", 2).over(journey_window)) \
    .withColumn("delay_s-2", F.lag("delay", 2).over(journey_window))

# Ersetze nur die null-Werte in den neuen Lag-2-Spalten
data_with_lags = data_with_lags.fillna({
    "occupancy_s-1": 0,  # falls du das auch möchtest
    "occupancy_s-2": 0,
    "delay_s-2": 0
})

# Ergebnis anzeigen
data_with_lags.show()

In [26]:
data_with_previous_occupancy = data_with_lags.filter(col("date") != "2025-04-12")

## 8. Berechnung von Referenzwerten in der Vergangenheit

In [27]:
data_with_previous_occupancy = data_with_previous_occupancy.withColumn(
    "month", month("date")
)

In [28]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, lag, round, col

time_intervals = ["interval_5min", "interval_15min", "interval_30min", "interval_60min"]
columns = ["occupancy_s-1", "boarders", "alighters"]

for col_name in columns:
    for time_interval in time_intervals:
        
        journey_window = (
            Window
            .partitionBy("month", "weekday", time_interval)
            .orderBy("actual_arrival")
        )
        
        avg_col_name = f"avg_{col_name}_{time_interval}"
        lag_col_name = f"lag_{col_name}_{time_interval}"

        data_with_previous_occupancy = data_with_previous_occupancy.withColumn(
            avg_col_name,
            round(avg(col_name).over(journey_window), 2)
        )

        data_with_previous_occupancy = data_with_previous_occupancy.withColumn(
            lag_col_name,
            lag(col_name, 1, -1).over(journey_window)  # <- default = 0
        )

data_with_prev_info = data_with_previous_occupancy


## 9. Vermerken der vorherigen und der nachfolgenden Haltestelle

In [29]:
journey_window = Window.partitionBy("device").orderBy("date", "actual_arrival")

# → Default für lag = "START", für lead = "END"
data_with_prev_info = data_with_prev_info.withColumn(
    "lag_stop_name", lag("stop_name", 1, "START").over(journey_window)
)

data_with_stop_names = data_with_prev_info.withColumn(
    "lead_stop_name", lead("stop_name", 1, "END").over(journey_window)
)

## 10. Finale Überprüfung der Datenkonsistenz

In [31]:
def check_missing_values(data_frame):
    total_rows = data_frame.count()  # Bestimme die Gesamtanzahl der Zeilen im DataFrame
    
    # Aggregiere die Anzahl der NULL-Werte für jede Spalte in einem Schritt
    missing_values = data_frame.select([
        F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in data_frame.columns
    ]).collect()[0].asDict()  # Konvertiere das Ergebnis in ein Dictionary

    # Konvertiere das Dictionary in einen DataFrame
    missing_df = spark.createDataFrame([
        (col_name, missing_count, total_rows) 
        for col_name, missing_count in missing_values.items()
    ], ["Column", "Missing Values", "Total Rows"]).orderBy(F.col("Missing Values").desc())  # Sortiere nach Anzahl fehlender Werte

    return missing_df

# Anwendung der Funktion
missing_info = check_missing_values(data_with_stop_names)
missing_info.show()

## 11. Speicherung der angereicherten Daten als Parquet-Datei.

In [32]:
# Pfad, unter dem die enrichierten Daten gespeichert werden sollen
path_enriched_data_df = "data/enriched_data"

# Speicherung des DataFrames als Parquet-Datei (overwrite-Modus)
data_with_stop_names.write.mode("overwrite").parquet(path_enriched_data_df)
print(f"Data written to {path_enriched_data_df} with {data_with_stop_names.count()} rows.")